# DNSMOS Pro scores for website examples

Run this notebook only after copying both sentence-47_01 Praat WAVs into
`outputs/web_examples/`. It requires the exact 11-file website manifest, scores
every sample, and generates matching spectrograms.

In [1]:
import os
import pandas as pd
from IPython.display import display

from dnsmos_common import (
    build_web_inventory, find_project_root, generate_web_spectrograms,
    score_web_examples,
)

PROJECT_ROOT = find_project_root()
DEVICE = os.environ.get("DNSMOS_DEVICE", "auto")
FORCE_RECOMPUTE = os.environ.get("DNSMOS_FORCE_RECOMPUTE", "false").lower() in {"1", "true", "yes"}

## 1. Validate the 11-file manifest

This check gives a clear missing/unexpected filename error before the model is
loaded.

In [2]:
inventory = build_web_inventory(PROJECT_ROOT)
assert len(inventory) == 11
display(inventory[["filename", "pipeline", "condition", "feature_set"]])
print("Manifest accepted: 11 unique website WAVs.")

,filename,pipeline,condition,feature_set
0,neu_original.wav,original,original,natural
1,hap_original.wav,original,original,natural
2,neu_reconstruct.wav,vowel_edit_pipeline,neu_reconstruct,reconstruction
3,neu-to-hap_pitch.wav,vowel_edit_pipeline,neu_to_hap,pitch
4,neu-to-hap_loudness.wav,vowel_edit_pipeline,neu_to_hap,loudness
5,neu-to-hap_periodicity.wav,vowel_edit_pipeline,neu_to_hap,periodicity
6,neu-to-hap_duration.wav,vowel_edit_pipeline,neu_to_hap,duration
7,neu-to-hap_ppg.wav,vowel_edit_pipeline,neu_to_hap,ppg
8,neu-to-hap_all.wav,vowel_edit_pipeline,neu_to_hap,all
9,praat_neu-to-hap.wav,praat_pipeline,neu_to_hap,pitch-loudness-duration


Manifest accepted: 11 unique website WAVs.


## 2. Compute individual DNSMOS Pro scores

In [3]:
scores = score_web_examples(
    PROJECT_ROOT, device=DEVICE, force_recompute=FORCE_RECOMPUTE
)
display(scores[["filename", "dnsmos_mos", "dnsmos_variance"]].style.format(
    {"dnsmos_mos": "{:.3f}", "dnsmos_variance": "{:.3f}"}
))
print(f"Saved: {PROJECT_ROOT / 'outputs' / 'dnsmos' / 'web_example_scores.csv'}")

web_example_scores.csv: 11 selected, 11 reused, 0 requiring inference on cpu


,filename,dnsmos_mos,dnsmos_variance
0,hap_original.wav,2.555,0.411
1,neu_original.wav,2.076,0.282
2,praat_neu-to-hap.wav,1.487,0.067
3,praat_neu-to-sad.wav,2.057,0.240
4,neu_reconstruct.wav,2.946,0.284
5,neu-to-hap_all.wav,2.854,0.317
6,neu-to-hap_duration.wav,2.756,0.357
7,neu-to-hap_loudness.wav,2.612,0.390
8,neu-to-hap_periodicity.wav,2.869,0.292
9,neu-to-hap_pitch.wav,2.843,0.348


Saved: C:\projects\ProMoNet\outputs\dnsmos\web_example_scores.csv


## 3. Generate comparable spectrograms

All images use the same dB range, frequency range, STFT settings, and
dimensions. Copies are written both beside the source examples and under the
static website image directory.

In [4]:
spectrograms = generate_web_spectrograms(PROJECT_ROOT)
assert len(spectrograms) == 11
for path in spectrograms:
    print(path.relative_to(PROJECT_ROOT))

outputs\web_examples\neu_original_spectrogram.png
outputs\web_examples\hap_original_spectrogram.png
outputs\web_examples\neu_reconstruct_spectrogram.png
outputs\web_examples\neu-to-hap_pitch_spectrogram.png
outputs\web_examples\neu-to-hap_loudness_spectrogram.png
outputs\web_examples\neu-to-hap_periodicity_spectrogram.png
outputs\web_examples\neu-to-hap_duration_spectrogram.png
outputs\web_examples\neu-to-hap_ppg_spectrogram.png
outputs\web_examples\neu-to-hap_all_spectrogram.png
outputs\web_examples\praat_neu-to-hap_spectrogram.png
outputs\web_examples\praat_neu-to-sad_spectrogram.png


## 4. Final acceptance checks

In [5]:
saved = pd.read_csv(
    PROJECT_ROOT / "outputs" / "dnsmos" / "web_example_scores.csv"
)
assert len(saved) == 11
assert saved["relative_path"].is_unique
assert saved["status"].eq("success").all()
assert saved[["dnsmos_mos", "dnsmos_variance"]].notna().all().all()
assert len(list(
    (PROJECT_ROOT / "docs" / "images" / "spectrograms").glob("*.png")
)) == 11
print("Acceptance checks passed: 11 scores and 11 mirrored spectrograms.")

Acceptance checks passed: 11 scores and 11 mirrored spectrograms.
